# Pendahuluan

Di dalam material ini, kamu akan mempelajari Data Heart Disease dan belajar bagaimana melakukan deployment. Lakukan kode dan jalankan setiap sel. Selama proses pembelajaran, kamu akan menemui beberapa blok **TODO** -- ikuti petunjuk untuk mengisinya sebelum menjalankan sel-sel tersebut dan melanjutkan.

Pada tutorial ini, kita akan membuat dashboard prediksi menggunakan Streamlit untuk dua kasus: prediksi Iris Species dan prediksi Heart Disease. Dashboard ini memungkinkan pengguna untuk memasukkan nilai fitur dan mendapatkan prediksi hasil berdasarkan model yang sudah dilatih sebelumnya.

**simpan data evaluasi**

In [20]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

# ---- Siapkan data ----
df = pd.read_csv("data.csv", index_col=0)
df = df.drop(columns=['education'])
df['sex'] = df['sex'].replace({'Male': 1, 'Female': 0})
df['income'] = df['income'].replace({'<=50K': 0, '>50K': 1})

mappings = {}
for col in ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'native.country']:
    mappings[col] = {val: i for i, val in enumerate(sorted(df[col].unique()))}
    df[col] = df[col].replace(mappings[col])

df = df.astype(int)
df = df[['education.num', 'age', 'hours.per.week', 'capital.gain', 'sex', 'relationship', 'income']]

X = df.loc[:, ~df.columns.isin(['income'])]
y = df['income']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print("Shape:", X_train.shape, X_test.shape, list(X.columns))

# ---- Tuning ----
def tuning(clf, grid):
    gs = GridSearchCV(estimator=clf, param_grid=grid, cv=5, n_jobs=-1, scoring='roc_auc')
    return gs.fit(X_train, y_train)

fit_clf_lg = tuning(LogisticRegression(random_state=42),
                    {'max_iter': [100, 500, 1000], 'solver': ['lbfgs', 'liblinear', 'saga']})
fit_clf_rf = tuning(RandomForestClassifier(random_state=42),
                    {'n_estimators': [100, 200, 300], 'max_depth': [5, 10, 15], 'criterion': ['gini', 'entropy']})
fit_clf_dt = tuning(DecisionTreeClassifier(random_state=42),
                    {'min_samples_leaf': [1, 5, 10, 20], 'max_depth': [3, 5, 7, 10], 'criterion': ['gini', 'entropy']})
fit_clf_mlp = tuning(MLPClassifier(max_iter=500, random_state=42),
                     {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'activation': ['relu', 'tanh'], 'solver': ['adam', 'sgd']})

for n, m in [("LR", fit_clf_lg), ("RF", fit_clf_rf), ("DT", fit_clf_dt), ("MLP", fit_clf_mlp)]:
    print(n, m.best_params_)

# ---- Probabilitas dan threshold terbaik ----
y_pred_logreg = fit_clf_lg.predict_proba(X_test)[:, 1]
y_pred_rf = fit_clf_rf.predict_proba(X_test)[:, 1]
y_pred_dt = fit_clf_dt.predict_proba(X_test)[:, 1]
y_pred_mlp = fit_clf_mlp.predict_proba(X_test)[:, 1]

def find_best_threshold(y_test, y_pred):
    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    return thresholds[np.argmax(tpr - fpr)]

best_threshold_logreg = find_best_threshold(y_test, y_pred_logreg)
best_threshold_rf = find_best_threshold(y_test, y_pred_rf)
best_threshold_dt = find_best_threshold(y_test, y_pred_dt)
best_threshold_mlp = find_best_threshold(y_test, y_pred_mlp)

# ---- Simpan eval_data.pkl ----
hasil_tuning = {
    'Logistic Regression': (fit_clf_lg, y_pred_logreg, best_threshold_logreg),
    'Decision Tree': (fit_clf_dt, y_pred_dt, best_threshold_dt),
    'Random Forest': (fit_clf_rf, y_pred_rf, best_threshold_rf),
    'MLP': (fit_clf_mlp, y_pred_mlp, best_threshold_mlp),
}

sebelum_tuning = {
    'Logistic Regression': {'accuracy': 0.8242, 'precision': 0.73, 'recall': 0.44, 'f1': 0.54},
    'Decision Tree':       {'accuracy': 0.8158, 'precision': 0.64, 'recall': 0.55, 'f1': 0.59},
    'Random Forest':       {'accuracy': 0.8251, 'precision': 0.65, 'recall': 0.58, 'f1': 0.62},
    'MLP':                 {'accuracy': 0.8445, 'precision': 0.73, 'recall': 0.57, 'f1': 0.64},
}

eval_data = {'before': sebelum_tuning, 'models': {}, 'n_test': int(len(y_test))}

for nama, (m, prob, thr) in hasil_tuning.items():
    rep = classification_report(y_test, m.predict(X_test), output_dict=True)
    y_thr = (prob >= thr).astype(int)
    rep_thr = classification_report(y_test, y_thr, output_dict=True)
    fpr, tpr, _ = roc_curve(y_test, prob)

    eval_data['models'][nama] = {
        'best_params': m.best_params_,
        'cv_auc': float(m.best_score_),
        'auc': float(roc_auc_score(y_test, prob)),
        'threshold': float(thr),
        'default': {'accuracy': rep['accuracy'], 'precision': rep['1']['precision'],
                    'recall': rep['1']['recall'], 'f1': rep['1']['f1-score']},
        'at_threshold': {'accuracy': rep_thr['accuracy'], 'precision': rep_thr['1']['precision'],
                         'recall': rep_thr['1']['recall'], 'f1': rep_thr['1']['f1-score']},
        'cm': confusion_matrix(y_test, y_thr).tolist(),
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist(),
    }

with open('eval_data.pkl', 'wb') as f:
    pickle.dump(eval_data, f)

print("Tersimpan: eval_data.pkl")
print(eval_data['models']['Random Forest']['at_threshold'])
print(eval_data['models']['Random Forest']['cm'])

Shape: (26029, 6) (6508, 6) ['education.num', 'age', 'hours.per.week', 'capital.gain', 'sex', 'relationship']
LR {'max_iter': 100, 'solver': 'liblinear'}
RF {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 300}
DT {'criterion': 'gini', 'max_depth': 7, 'min_samples_leaf': 20}
MLP {'activation': 'relu', 'hidden_layer_sizes': (100,), 'solver': 'adam'}
Tersimpan: eval_data.pkl
{'accuracy': 0.8182237246465888, 'precision': 0.5877792977656179, 'recall': 0.8220663265306123, 'f1': 0.6854559957458123}
[[4036, 904], [279, 1289]]


**import**

In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
import time
import matplotlib.pyplot as plt

Overwriting app.py


**konfigurasi, muat model, dan konstanta**

In [22]:

%%writefile -a app.py

st.set_page_config(page_title="Adult Income Prediction", layout="wide")

# ---- Custom CSS ----
st.markdown("""
<style>

/* Ukuran teks dasar */
html {
    font-size: 16px !important;
}

[data-testid="stAppViewContainer"] {
    font-size: 16px !important;
}

/* Judul halaman */
h1 {
    font-size: 2.2rem !important;
    font-weight: 700 !important;
    line-height: 1.3 !important;
}

h2 {
    font-size: 1.7rem !important;
    font-weight: 650 !important;
}

h3 {
    font-size: 1.35rem !important;
    font-weight: 600 !important;
}

/* Teks paragraf */
[data-testid="stMarkdownContainer"] p {
    font-size: 1rem !important;
    line-height: 1.6 !important;
}

/* Sidebar */
[data-testid="stSidebar"] {
    font-size: 16px !important;
}

[data-testid="stSidebar"] p,
[data-testid="stSidebar"] label {
    font-size: 1rem !important;
}

/* Metric */
[data-testid="stMetricLabel"] {
    font-size: 1rem !important;
}

[data-testid="stMetricValue"] {
    font-size: 2.2rem !important;
    font-weight: 600 !important;
}

/* Caption */
[data-testid="stCaptionContainer"] {
    font-size: 0.85rem !important;
}

/* Tombol */
.stButton button {
    font-size: 1rem !important;
    padding: 0.5rem 1.2rem !important;
}

/* Input */
.stSelectbox label,
.stSlider label,
.stNumberInput label,
.stFileUploader label {
    font-size: 1rem !important;
}

/* Tab */
[data-testid="stTab"] {
    font-size: 1rem !important;
}

/* Tabel */
[data-testid="stDataFrame"] {
    font-size: 0.95rem !important;
}

</style>
""", unsafe_allow_html=True)


REPO_URL = "https://github.com/wulanlumbantoruan/Capstone-Project-PSD"
FITUR_WAJIB = ['education.num', 'age', 'hours.per.week', 'capital.gain', 'sex', 'relationship']


@st.cache_resource
def load_artifact():
    with open("generate_income.pkl", "rb") as file:
        return pickle.load(file)


@st.cache_resource
def load_eval():
    with open("eval_data.pkl", "rb") as file:
        return pickle.load(file)


@st.cache_data
def load_dataset():
    try:
        return pd.read_csv("adult.csv")
    except FileNotFoundError:
        st.error(f"File adult.csv tidak ditemukan. Silakan akses data di repositori: {REPO_URL}")
        st.stop()


artifact = load_artifact()
model = artifact['model']
scaler = artifact['scaler']
mappings = artifact['mappings']
features = artifact['features']
threshold = artifact['threshold']
ev = load_eval()


Appending to app.py


**Home / Dashboard**

In [23]:
%%writefile -a app.py


def page_home():
    st.title("Capstone Project - Kelompok 6 PSD")
    st.header("Adult Income Prediction")
    st.markdown(f"**GitHub:** [{REPO_URL}]({REPO_URL})")

    st.subheader("Informasi dataset")
    st.write("Dataset Adult Census Income berisi data sensus penduduk Amerika Serikat tahun 1994. "
             "Tujuannya memprediksi apakah pendapatan tahunan seseorang **di atas 50K USD (>50K)** "
             "atau **tidak (<=50K)** berdasarkan atribut demografi dan pekerjaan.")

    df = load_dataset()
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Jumlah data", f"{len(df):,}")
    c2.metric("Jumlah fitur", df.shape[1] - 1)
    c3.metric("Jumlah kelas", df['income'].nunique())
    c4.metric("Fitur dipakai model", len(FITUR_WAJIB))
    st.caption("Setelah 24 data duplikat dihapus, 32.537 data digunakan: 26.029 data latih dan "
               "6.508 data uji (rasio 80:20).")

    st.write("Jumlah data per kelas:")
    st.bar_chart(df['income'].value_counts())

    st.subheader("Algoritma yang digunakan")
    st.write("Empat algoritma dibandingkan: **Logistic Regression**, **Decision Tree**, "
             "**Random Forest**, dan **MLP (Neural Network)**, masing-masing dengan hyperparameter tuning. "
             "Model terbaik yang dipakai pada aplikasi ini adalah **Random Forest**.")

    st.subheader("Ringkasan performa model (Random Forest)")
    rf = ev['models']['Random Forest']
    t = rf['at_threshold']
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("AUC-ROC", f"{rf['auc']:.3f}")
    m2.metric("Akurasi", f"{t['accuracy']:.3f}")
    m3.metric("Recall (>50K)", f"{t['recall']:.3f}")
    m4.metric("F1-score (>50K)", f"{t['f1']:.3f}")
    st.caption(f"Dihitung pada data uji dengan threshold terbaik {rf['threshold']:.3f}. "
               "Detail perbandingan semua model ada di halaman Model Evaluation.")

Appending to app.py


**Prediction**

In [24]:
%%writefile -a app.py


def hasil_prediksi(input_df):
    X_in = input_df[features]
    X_scaled = scaler.transform(X_in)
    proba = model.predict_proba(X_scaled)[:, 1]
    pred = (proba >= threshold).astype(int)
    return proba, pred


def page_prediction():
    st.title("Prediction")
    st.write("Masukkan data seseorang untuk memprediksi kelas pendapatannya.")
    tab_manual, tab_csv = st.tabs(["Input manual", "Unggah CSV"])

    with tab_manual:
        c1, c2 = st.columns(2)
        with c1:
            education_num = st.slider("Tingkat pendidikan (education.num)", 1, 16, 10)
            st.caption("Contoh: 9 = HS-grad, 10 = Some-college, 13 = Bachelors, 14 = Masters, 16 = Doctorate")
            age = st.slider("Usia (age)", 17, 90, 35)
            hours = st.slider("Jam kerja per minggu (hours.per.week)", 1, 99, 40)
        with c2:
            capital_gain = st.number_input("Capital gain (USD)", min_value=0, max_value=99999, value=0)
            sex_label = st.selectbox("Jenis kelamin (sex)", ('Perempuan', 'Pria'))
            rel_label = st.selectbox("Peran dalam keluarga (relationship)", list(mappings['relationship'].keys()))

        if st.button("Predict!"):
            input_df = pd.DataFrame({
                'education.num': [education_num],
                'age': [age],
                'hours.per.week': [hours],
                'capital.gain': [capital_gain],
                'sex': [0 if sex_label == 'Perempuan' else 1],
                'relationship': [mappings['relationship'][rel_label]],
            })
            proba, pred = hasil_prediksi(input_df)

            st.subheader("Hasil prediksi")
            if pred[0] == 1:
                st.success("Kelas: **>50K** (pendapatan tahunan di atas 50.000 USD)")
            else:
                st.info("Kelas: **<=50K** (pendapatan tahunan 50.000 USD atau kurang)")
            st.write(f"Probabilitas kelas >50K: **{proba[0]:.2f}**")
            st.progress(float(proba[0]))

            st.subheader("Keterangan hasil klasifikasi")
            st.write(f"Model Random Forest menghitung probabilitas kelas >50K, lalu memprediksi **>50K** "
                     f"jika probabilitasnya mencapai threshold **{threshold:.3f}** (bukan 0,5). "
                     "Threshold diturunkan agar lebih banyak orang berpenghasilan >50K terdeteksi "
                     "(recall naik), dengan konsekuensi precision lebih rendah. "
                     "Data latih berasal dari sensus 1994, sehingga hasil ini bersifat indikatif.")
            st.write("Data yang dimasukkan:")
            st.dataframe(pd.DataFrame({
                'Fitur': ['education.num', 'age', 'hours.per.week', 'capital.gain', 'sex', 'relationship'],
                'Nilai': [education_num, age, hours, capital_gain, sex_label, rel_label]}))

    with tab_csv:
        template = pd.DataFrame({
            'education.num': [13, 9],
            'age': [39, 50],
            'hours.per.week': [40, 45],
            'capital.gain': [0, 5000],
            'sex': ['Male', 'Female'],
            'relationship': ['Husband', 'Not-in-family']
        })
        st.download_button("Download template CSV", template.to_csv(index=False),
                           file_name="template_input.csv", mime="text/csv")
        st.caption(f"Nilai relationship yang valid: {list(mappings['relationship'].keys())}")
        uploaded_file = st.file_uploader("Upload your input CSV file", type=["csv"])

        if uploaded_file is not None:
            input_df = pd.read_csv(uploaded_file)
            kurang = [c for c in FITUR_WAJIB if c not in input_df.columns]
            if kurang:
                st.error(f"Kolom berikut belum ada di CSV: {kurang}. Gunakan template CSV di atas.")
                return
            if not pd.api.types.is_numeric_dtype(input_df['sex']):
                input_df['sex'] = input_df['sex'].map({'Female': 0, 'Male': 1})
            if not pd.api.types.is_numeric_dtype(input_df['relationship']):
                input_df['relationship'] = input_df['relationship'].map(mappings['relationship'])
            if input_df[FITUR_WAJIB].isnull().any().any():
                st.error("Ada nilai kosong atau kategori tidak dikenali pada kolom 'sex' atau 'relationship'.")
                return

            if st.button("Predict CSV!"):
                proba, pred = hasil_prediksi(input_df)
                out = input_df.copy()
                out['probabilitas >50K'] = proba.round(3)
                out['prediksi'] = ['>50K' if p == 1 else '<=50K' for p in pred]
                st.dataframe(out)

Appending to app.py


**Model Evaluation**

In [25]:
%%writefile -a app.py


def page_evaluation():
    st.title("Model Evaluation")
    st.write(f"Evaluasi pada data uji ({ev['n_test']:,} data). Metrik precision, recall, dan F1-score "
             "dihitung untuk kelas >50K, karena data target tidak seimbang (76% <=50K dan 24% >50K).")
    nama_model = list(ev['models'].keys())

    st.subheader("1. Perbandingan model sebelum dan sesudah hyperparameter tuning")
    baris = []
    for n in nama_model:
        b = ev['before'][n]
        a = ev['models'][n]['default']
        baris.append({'Model': n,
                      'Akurasi (sebelum)': b['accuracy'], 'Akurasi (sesudah)': a['accuracy'],
                      'Recall (sebelum)': b['recall'], 'Recall (sesudah)': a['recall'],
                      'F1 (sebelum)': b['f1'], 'F1 (sesudah)': a['f1']})
    st.dataframe(pd.DataFrame(baris).round(3).set_index('Model'))
    st.caption("Sebelum dan sesudah tuning memakai threshold bawaan 0,5.")

    st.subheader("2. Parameter terbaik (GridSearchCV, cv=5, scoring=roc_auc)")
    st.dataframe(pd.DataFrame({
        'Model': nama_model,
        'Best params': [str(ev['models'][n]['best_params']) for n in nama_model],
        'AUC validasi silang': [round(ev['models'][n]['cv_auc'], 3) for n in nama_model]}).set_index('Model'))

    st.subheader("3. ROC dan AUC")
    fig, ax = plt.subplots(figsize=(7, 5))
    for n in nama_model:
        ax.plot(ev['models'][n]['fpr'], ev['models'][n]['tpr'], label=f"{n} (AUC = {ev['models'][n]['auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves for Income Prediction Models')
    ax.legend()
    st.pyplot(fig)

    st.subheader("4. Threshold terbaik (Youden's Index) dan performanya")
    baris = []
    for n in nama_model:
        m = ev['models'][n]
        t = m['at_threshold']
        baris.append({'Model': n, 'AUC': m['auc'], 'Threshold terbaik': m['threshold'],
                      'Akurasi': t['accuracy'], 'Precision': t['precision'],
                      'Recall': t['recall'], 'F1': t['f1']})
    st.dataframe(pd.DataFrame(baris).round(3).set_index('Model'))

    st.subheader("5. Confusion matrix Random Forest (threshold terbaik)")
    cm = ev['models']['Random Forest']['cm']
    st.dataframe(pd.DataFrame(cm, index=['Aktual <=50K', 'Aktual >50K'],
                              columns=['Prediksi <=50K', 'Prediksi >50K']))

    st.subheader("6. Kesimpulan")
    rf = ev['models']['Random Forest']
    t = rf['at_threshold']
    st.write(f"Model **Random Forest** dipilih karena memiliki AUC-ROC tertinggi ({rf['auc']:.3f}) "
             f"dan performa paling seimbang pada threshold terbaik ({rf['threshold']:.3f}): "
             f"recall {t['recall']:.2f}, precision {t['precision']:.2f}, F1-score {t['f1']:.2f}, "
             f"dan akurasi {t['accuracy']:.2f}. Pada threshold bawaan 0,5, recall kelas >50K hanya "
             f"{rf['default']['recall']:.2f}, sehingga threshold diturunkan agar lebih banyak responden "
             "berpenghasilan >50K terdeteksi, dengan konsekuensi precision lebih rendah.")

Appending to app.py


**Dataset Information**

In [26]:
%%writefile -a app.py


FITUR_DESKRIPSI = pd.DataFrame([
    ['age', 'Numerik', 'Usia responden (tahun)', 'Ya'],
    ['workclass', 'Kategorik', 'Sektor tempat bekerja (Private, Self-emp, Federal-gov, dst.)', 'Tidak'],
    ['fnlwgt', 'Numerik', 'Bobot sampel sensus: perkiraan jumlah penduduk yang diwakili responden', 'Tidak'],
    ['education', 'Kategorik', 'Jenjang pendidikan tertinggi (teks)', 'Tidak (diwakili education.num)'],
    ['education.num', 'Numerik (ordinal)', 'Jenjang pendidikan dalam angka 1-16 (9 = HS-grad, 13 = Bachelors, 16 = Doctorate)', 'Ya'],
    ['marital.status', 'Kategorik', 'Status pernikahan', 'Tidak'],
    ['occupation', 'Kategorik', 'Jenis pekerjaan', 'Tidak'],
    ['relationship', 'Kategorik', 'Peran responden dalam keluarga (Husband, Wife, Own-child, dst.)', 'Ya'],
    ['race', 'Kategorik', 'Ras', 'Tidak'],
    ['sex', 'Kategorik', 'Jenis kelamin', 'Ya'],
    ['capital.gain', 'Numerik', 'Keuntungan modal dari investasi di luar gaji (USD)', 'Ya'],
    ['capital.loss', 'Numerik', 'Kerugian modal (USD)', 'Tidak'],
    ['hours.per.week', 'Numerik', 'Jam kerja per minggu', 'Ya'],
    ['native.country', 'Kategorik', 'Negara asal', 'Tidak'],
    ['income', 'Kategorik (target)', 'Kelas pendapatan tahunan: <=50K atau >50K', 'Target'],
], columns=['Fitur', 'Tipe', 'Deskripsi', 'Dipakai model'])


def page_dataset():
    st.title("Dataset Information")
    st.markdown(f"Akses data di repositori GitHub: [{REPO_URL}]({REPO_URL})")
    df = load_dataset()
    st.write(f"Dataset memiliki **{df.shape[0]:,} baris** dan **{df.shape[1]} kolom**. "
             "Nilai tidak diketahui pada data asli ditulis dengan tanda '?' (kolom workclass, "
             "occupation, dan native.country).")

    st.subheader("Preview dataset")
    n = st.slider("Jumlah baris ditampilkan", 5, 100, 10)
    st.dataframe(df.head(n))

    st.subheader("Nama dan deskripsi fitur")
    st.dataframe(FITUR_DESKRIPSI.set_index('Fitur'))

Appending to app.py


**navigasi**

In [27]:
%%writefile -a app.py


st.sidebar.title("Menu")
halaman = st.sidebar.radio("Pilih halaman",
                           ["Home / Dashboard", "Prediction", "Model Evaluation", "Dataset Information"])

if halaman == "Home / Dashboard":
    page_home()
elif halaman == "Prediction":
    page_prediction()
elif halaman == "Model Evaluation":
    page_evaluation()
else:
    page_dataset()

Appending to app.py
